In [7]:
from google.colab import files
import os
import pandas as pd

#อัปโหลดไฟล์
ref_path = '/content/Sony_Price_REF.csv'
data_path = '/content/Dataset Sony Camera(Lens not Included).csv'

if not os.path.exists(ref_path) or not os.path.exists(data_path):
    print('กรุณาอัปโหลดไฟล์ Sony_Price_REF.csv และ Dataset Sony Camera(Lens not Included).csv')
    uploaded = files.upload()

if os.path.exists(ref_path) and os.path.exists(data_path):
    ref_df = pd.read_csv(ref_path)
    ref_df['score'] = 10
    df = pd.read_csv(data_path)

    display(ref_df.head())
    display(df.head())
    print(f'Reference rows: {len(ref_df)}')
    print(f'Dataset rows: {len(df)}')
else:
    print('Files still missing. Please ensure they are uploaded to /content/')


,model,release_year,age_year,shutter_claim,condition,price_used,score
0,Sony A5000,2014,12,100000,good,5000,10
1,Sony A5100,2014,12,100000,good,6500,10
2,Sony A6000,2014,12,100000,good,8000,10
3,Sony A6100,2019,7,100000,very_good,16500,10
4,Sony A6300,2016,10,100000,good,11500,10


,model,age_year,shutter_count,condition,has_box,price_used
0,Sony A6600,0,42098,very_good,0,24390
1,Sony A5100,1,8379,very_good,1,8000
2,Sony A7SIII,6,1026,excellent,0,61190
3,Sony A7RV,0,1684,very_good,0,97700
4,Sony A6600,6,4903,very_good,0,26100


Reference rows: 35
Dataset rows: 5000


In [8]:
import pandas as pd
import numpy as np
import os
import re

ref_path = '/content/Sony_Price_REF.csv'
data_path = '/content/Dataset Sony Camera(Lens not Included).csv'

if os.path.exists(ref_path) and os.path.exists(data_path):
    ref_df = pd.read_csv(ref_path)
    df = pd.read_csv(data_path)

    if 'score' not in ref_df.columns:
        ref_df['score'] = 10

    df['condition'] = df['condition'].astype(str).str.lower()
    df['shutter_count'] = pd.to_numeric(df['shutter_count'], errors='coerce').fillna(0)
    df['age_year'] = pd.to_numeric(df['age_year'], errors='coerce').fillna(0)

    ref_df['model_clean'] = ref_df['model'].str.replace('Sony ', '', regex=False)
    df['model_clean'] = df['model'].str.replace('Sony ', '', regex=False)

    merged_df = pd.merge(df, ref_df[["model_clean", "price_used", "score"]], on="model_clean", how="left", suffixes=("", "_ref"))

    def calculate_scores(row):
        scores = {}
        cond_map = {"excellent": 10, "very_good": 8, "good": 6, "fair": 4, "poor": 2}
        scores["cond_score"] = cond_map.get(row["condition"], 5)
        scores["age_score"] = 10 if row["age_year"] < 3 else 5
        scores["shutter_score"] = 10 if row["shutter_count"] < 20000 else 5

        try:
            p_main = float(re.sub(r"[^0-9.]", "", str(row["price_used"])))
            p_ref = float(re.sub(r"[^0-9.]", "", str(row["price_used_ref"])))
            scores["price_score"] = 10 if p_main <= p_ref else 5
        except:
            scores["price_score"] = 5

        scores["box_score"] = 10 if row["has_box"] == 1 else 0
        return pd.Series(scores)

    score_cols = merged_df.apply(calculate_scores, axis=1)
    final_df = pd.concat([merged_df, score_cols], axis=1)
    print("Data processing complete. final_df is ready.")
else:
    print("Error: CSV files missing in /content/")

Data processing complete. final_df is ready.


In [24]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

def clean_val(val):
    try:
        res = float(str(val).replace(',', '').replace('-', '0'))
        return res if np.isfinite(res) else 0.0
    except:
        return 0.0

y_price_train_clean = y_train.apply(clean_val).fillna(0.0)
y_price_test_clean = y_test.apply(clean_val).fillna(0.0)

X_train_numeric = X_train.apply(pd.to_numeric, errors='coerce').fillna(0.0)
X_test_numeric = X_test.apply(pd.to_numeric, errors='coerce').fillna(0.0)

price_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42)
}

print('--- Price Prediction Model Results ---')
for name, model in price_models.items():
    model.fit(X_train_numeric, y_price_train_clean)
    preds = model.predict(X_test_numeric)
    r2 = r2_score(y_price_test_clean, preds)
    mae = mean_absolute_error(y_price_test_clean, preds)
    print(f'{name}: R2 Score = {r2:.4f}, MAE = {mae:.2f}')

--- Price Prediction Model Results ---
Linear Regression: R2 Score = 0.9588, MAE = 4743.31
Random Forest Regressor: R2 Score = 0.9762, MAE = 2016.73
Gradient Boosting Regressor: R2 Score = 0.9788, MAE = 2105.02


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
purchase_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest Classifier': RandomForestClassifier(random_state=42),
    'Gradient Boosting Classifier': GradientBoostingClassifier(random_state=42)
}

print('--- Purchase Prediction Accuracy Results ---')
for name, model in purchase_models.items():
    model.fit(X_train_numeric, y_purch_train)
    preds = model.predict(X_test_numeric)
    acc = accuracy_score(y_purch_test, preds)
    print(f'{name} Accuracy: {acc*100:.2f}%')

In [6]:
from google.colab import files
import os

print('กรุณาอัปโหลดไฟล์ Sony_Price_REF.csv และ Dataset Sony Camera(Lens not Included).csv')
uploaded = files.upload()

required_files = ['Sony_Price_REF.csv', 'Dataset Sony Camera(Lens not Included).csv']
for f in required_files:
    if os.path.exists(f):
        print(f'✅ {f} อัปโหลดเรียบร้อย')
    else:
        print(f'❌ ไม่พบไฟล์ {f} กรุณาอัปโหลดใหม่อีกครั้ง')

กรุณาอัปโหลดไฟล์ Sony_Price_REF.csv และ Dataset Sony Camera(Lens not Included).csv


Saving Dataset Sony Camera(Lens not Included).csv to Dataset Sony Camera(Lens not Included).csv
Saving Sony_Price_REF.csv to Sony_Price_REF.csv
✅ Sony_Price_REF.csv อัปโหลดเรียบร้อย
✅ Dataset Sony Camera(Lens not Included).csv อัปโหลดเรียบร้อย


In [25]:
import joblib
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder

if 'final_df' in globals():
    le = LabelEncoder()
    final_df['model_encoded'] = le.fit_transform(final_df['model_clean'])
    joblib.dump(le, 'model_encoder.joblib')

    def get_num(x):
        try: return float(re.sub(r'[^0-9.]', '', str(x)))
        except: return 0.0

    final_df['price_used_ref_numeric'] = final_df['price_used_ref'].apply(get_num)

    # Features setup
    features = ['model_encoded', 'price_used_ref_numeric', 'age_year', 'shutter_count', 'has_box', 'cond_score', 'age_score', 'shutter_score', 'box_score']
    X_ml = final_df[features].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    y_price = final_df['price_used'].apply(get_num)

    X_train, X_test, y_train, y_test = train_test_split(X_ml, y_price, test_size=0.15, random_state=42)

    final_model = VotingRegressor([
        ('rf', RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42)),
        ('gb', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42)),
        ('ridge', Ridge(alpha=1.0)),
        ('et', ExtraTreesRegressor(n_estimators=300, random_state=42))
    ])

    final_model.fit(X_train, y_train)
    joblib.dump(final_model, 'Sony_Camera_Predict.joblib')
    print('Model calibrated for specific model hierarchies and market grounding.')
else:
    print('Error: final_df missing.')

Model calibrated for specific model hierarchies and market grounding.


In [26]:
import joblib
import pandas as pd
import os
import re

model_path = 'Sony_Camera_Predict.joblib'
ref_path = '/content/Sony_Price_REF.csv'
encoder_path = 'model_encoder.joblib'

if os.path.exists(model_path) and os.path.exists(ref_path) and os.path.exists(encoder_path):
    model = joblib.load(model_path)
    ref_df = pd.read_csv(ref_path)
    le = joblib.load(encoder_path)

    print('--- ผลการทำนายราคา AI ---')
    header = f'{"Model Name":<20} | {"Scenario":<15} | {"Predicted Price":<15}'
    print(header)
    print('-' * len(header))

    for _, row in ref_df.iterrows():
        m_name = row['model']
        m_clean = m_name.replace('Sony ', '')
        ref_p = float(re.sub(r'[^0-9.]', '', str(row['price_used'])))

        try:
            m_idx = le.transform([m_clean])[0]
        except:
            m_idx = 0

        case_mint = [m_idx, ref_p, 1, 500, 1, 10, 10, 10, 10]
        case_used = [m_idx, ref_p, max(row['age_year'], 3), 50000, 0, 6, 5, 5, 0]

        test_df = pd.DataFrame([case_mint, case_used], columns=['model_encoded', 'price_used_ref_numeric', 'age_year', 'shutter_count', 'has_box', 'cond_score', 'age_score', 'shutter_score', 'box_score'])
        preds = model.predict(test_df)
        mint_p = preds[0]
        mint_p = (mint_p * 0.3) + (ref_p * 0.7)
        used_p = min(preds[1], mint_p * 0.80)

        print(f'{m_name:<20} | {"1. Mint":<15} | {mint_p:,.2f} บาท')
        print(f'{m_name:<20} | {"2. Used":<15} | {used_p:,.2f} บาท')
        print('.' * len(header))
else:
    print('Please run the training cell first.')

--- ผลการทำนายราคา AI ---
Model Name           | Scenario        | Predicted Price
--------------------------------------------------------
Sony A5000           | 1. Mint         | 5,671.66 บาท
Sony A5000           | 2. Used         | 3,949.72 บาท
........................................................
Sony A5100           | 1. Mint         | 7,337.23 บาท
Sony A5100           | 2. Used         | 5,869.79 บาท
........................................................
Sony A6000           | 1. Mint         | 8,655.39 บาท
Sony A6000           | 2. Used         | 6,924.31 บาท
........................................................
Sony A6100           | 1. Mint         | 17,093.45 บาท
Sony A6100           | 2. Used         | 13,674.76 บาท
........................................................
Sony A6300           | 1. Mint         | 12,217.10 บาท
Sony A6300           | 2. Used         | 9,773.68 บาท
........................................................
Sony A6400           | 1. Mint  

In [29]:
from google.colab import files
import os

files_to_download = ['Sony_Camera_Predict.joblib', 'model_encoder.joblib']

print('--- กำลังตรวจสอบและเตรียมดาวน์โหลดโมเดลล่าสุด ---')

found_any = False
for f in files_to_download:
    if os.path.exists(f):
        print(f'✅ กำลังดาวน์โหลด: {f}')
        files.download(f)
        found_any = True
    else:
        print(f'❌ ไม่พบไฟล์: {f}')

if not found_any:
    print('\nรันเซลล์การเทรน (c21afac8) ก่อน')

--- กำลังตรวจสอบและเตรียมดาวน์โหลดโมเดลล่าสุด ---
✅ กำลังดาวน์โหลด: Sony_Camera_Predict.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ กำลังดาวน์โหลด: model_encoder.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>